# spaCy : Text Classification & Model Training
 - This is "classification" in the conventional machine learning sense, and it is applied to text. 
 - Examples include spam detection, sentiment analysis, and tagging customer queries etc

In [3]:
import pandas as pd

# Loading the spam data
# ham is the label for non-spam messages
spam = pd.read_csv('../resources/spam.csv')
spam.head(10)

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


## Bag of Words
 - Machine learning models don't learn from raw text data. Instead, you need to convert the text to something numeric.
 - The simplest common representation is a variation of one-hot encoding where each document as a vector of term frequencies for each term in the vocabulary.
 - *The vocabulary* is built from all the tokens (terms) in the corpus (the collection of documents).
 - **Example** : "Tea is life. Tea is love." and "Tea is healthy, calming, and delicious." as our corpus. 
   *The vocabulary* then is {"tea", "is", "life", "love", "healthy", "calming", "and", "delicious"} (ignoring punctuation).
   
 - *bag-of-words* is a representation of text that describes the occurrence of words within a document. It involves two things:
 - **A vocabulary of known words.**
 - **A measure of the presence of known words.**
 
 - It is called a “bag” of words, because any information about the order or structure of words in the document is discarded.      The model is only concerned with whether known words occur in the document, not where in the document.
 - The intuition is that documents are similar if they have similar content
 
 -For each document, count up how many times a term occurs, and place that count in the appropriate element of a vector. The  first sentence has "tea" twice and that is the first position in our vocabulary, so we put the number 2 in the first element of the vector. Our sentences as vectors then look like

 v1= [22110000]
 
This is called the bag of words representation. You can see that documents with similar terms will have similar vectors.
 
 - Ref: https://machinelearningmastery.com/gentle-introduction-bag-words-model/

## Model building : bag of words
 - Once you have your documents in a bag of words representation, you can use those vectors as input to any machine learning model. 
 - spaCy handles the bag of words conversion and building a simple linear model for you with the TextCategorizer class.

In [5]:
#we'll create a TextCategorizer pipe and add it to the empty model.
#Pipes are classes for processing and transforming tokens.

#Since the classes are either ham or spam, we set "exclusive_classes" to True.
#We've also configured it with the bag of words ("bow") architecture
import spacy

# Create an empty model
nlp = spacy.blank("en")

# Create the TextCategorizer with exclusive classes and "bow" architecture
textcat = nlp.create_pipe(
              "textcat",
              config={
                "exclusive_classes": True,
                "architecture": "bow"})

# Add the TextCategorizer to the empty model
nlp.add_pipe(textcat)

In [6]:
# Add labels to text classifier
textcat.add_label("ham")
textcat.add_label("spam")

1

## Model Training : Text Categorizer Model preparation
- convert the labels in the data to the form TextCategorizer requires. For each document,create a dictionary of boolean values for each class.
- Then we combine the texts and labels into a single list.
- python zip: Join two tuples together The zip() function returns a zip object, which is an iterator of tuples where the first item in each passed iterator is paired together

In [7]:
train_texts = spam['text'].values
train_labels = [{'cats': {'ham': label == 'ham',
                          'spam': label == 'spam'}} 
                for label in spam['label']]

train_data = list(zip(train_texts, train_labels))
train_data[:3]

[('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...',
  {'cats': {'ham': True, 'spam': False}}),
 ('Ok lar... Joking wif u oni...', {'cats': {'ham': True, 'spam': False}}),
 ("Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
  {'cats': {'ham': False, 'spam': True}})]

## Model Training : Optimizer & begin training
- First, create an optimizer using nlp.begin_training()  spaCy uses this optimizer to update the model. 
- it's more efficient to train models in small batches. spaCy provides the minibatch function that returns a generator yielding minibatches for training.
- Finally, the minibatches are split into texts and labels, then used with nlp.update to update the model's parameters.

In [8]:
from spacy.util import minibatch

spacy.util.fix_random_seed(1)
optimizer = nlp.begin_training()

# Create the batch generator with batch size = 8
batches = minibatch(train_data, size=8)
# Iterate through minibatches
for batch in batches:
    # Each batch is a list of (text, label) but we need to
    # send separate lists for texts and labels to update().
    # This is a quick way to split a list of tuples into lists
    texts, labels = zip(*batch)
    nlp.update(texts, labels, sgd=optimizer)

In [9]:
# This is just one training loop (or epoch) through the data. The model will typically need multiple epochs. 
# Use another loop for more epochs, and optionally re-shuffle the training data at the begining of each loop.

import random

random.seed(1)
spacy.util.fix_random_seed(1)
optimizer = nlp.begin_training()

losses = {}
for epoch in range(10):
    random.shuffle(train_data)
    # Create the batch generator with batch size = 8
    batches = minibatch(train_data, size=8)
    # Iterate through minibatches
    for batch in batches:
        # Each batch is a list of (text, label) but we need to
        # send separate lists for texts and labels to update().
        # This is a quick way to split a list of tuples into lists
        texts, labels = zip(*batch)
        nlp.update(texts, labels, sgd=optimizer, losses=losses)
    print(losses)

{'textcat': 0.43189741921099767}
{'textcat': 0.6474976215331196}
{'textcat': 0.7842154536487618}
{'textcat': 0.8716683716818165}
{'textcat': 0.9280939335008995}
{'textcat': 0.9655779922872296}
{'textcat': 0.9939651840090362}
{'textcat': 1.0127976631523663}
{'textcat': 1.0275637812859075}
{'textcat': 1.0378531470013608}


## Making Predictions

- We can make predictions with the predict() method. 
- The input text needs to be tokenized with nlp.tokenizer. Then you pass the tokens to the predict method which returns scores. 
- The scores are the probability the input text belongs to the classes.

In [12]:
texts = ["URGENT Reply to this message for GUARANTEED FREE TEA" ]
docs = [nlp.tokenizer(text) for text in texts]
    
# Use textcat to get the scores for each doc
textcat = nlp.get_pipe('textcat')
scores, _ = textcat.predict(docs)

print(scores)

[[0.01149131 0.98850864]]


In [13]:
# The scores are used to predict a single class or label by choosing the label with the highest probability. 
# We can get the index of the highest probability with scores.argmax, then use the index to get the label string from textcat.labels.
# From the scores, find the label with the highest score/probability
predicted_labels = scores.argmax(axis=1)
print([textcat.labels[label] for label in predicted_labels])

['spam']
